# 03: Python-based Preprocessing Routine Reproducibility

## Dataset
- [GSE223065](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE223065)

---
## 0. Execution Environment Summary

In [ ]:
%%bash
bash .rep_exe_env_short.sh

---
## 1. Data Download

In [ ]:
%%bash

DATA_DIR="./data"
FILE_NAME="gbm"
SRC="https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE223065&format=file"

mkdir -p "$DATA_DIR/"

if [ -f "$DATA_DIR/$FILE_NAME/*.gz" ]; then
    echo "$FILE_NAME data already exists. Skipping download"
else
    curl -o "$DATA_DIR/$_NAME/data.tar" $SRC
    tar -xvf "$DATA_DIR/$FILE_NAME/data.tar" -C $FILE_NAME
    rm -rf "$DATA_DIR/$FILE_NAME/data.tar"
fi

---
## 2. QC
### 2-1. Thresholding QC metrics

In [ ]:
import glob
import os

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns

from basalcelldemo_tools.anndata import transfer_embedding_info
from basalcelldemo_tools.preferences import (
    DATA_DIR,
    OUTPUT_DIR,
    kwarg_savefig,
)

ENV_SUFFIX = os.environ["ENV_SUFFIX"]

In [ ]:

mtx_files = glob.glob("ucsf/*_matrix.mtx.gz")
mtx_prefix = [v.split("/")[-1].split("matrix.mtx")[0] for v in mtx_files]

datasets = {}

for prefix in mtx_prefix:
    key = prefix.split("_")[1]
    datasets[key] = sc.read_10x_mtx(
        DATA_DIR / "gbm", prefix=prefix
    )

In [ ]:
for k, _adata in datasets.items():
    _adata.var["mt"] = _adata.var_names.str.startswith("MT-")
    _adata.var["ribo"] = _adata.var_names.str.startswith(("RPS", "RPL"))
    _adata.var["hb"] = _adata.var_names.str.contains("^HB[^(P)]")
    sc.pp.calculate_qc_metrics(
        _adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
    )
    sc.pl.violin(
        _adata,
        [
            "n_genes_by_counts", "total_counts",
            "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"
        ],
        jitter=0.4,
        multi_panel=True,
    )
    datasets[k] = _adata[_adata.obs.pct_counts_mt < 25, :].copy()

### 2-2. Doublet Detection

In [ ]:
for k, _adata in datasets.items():
    sc.pp.scrublet(_adata)
    sc.pl.scatter(
        _adata, x="total_counts", y="n_genes_by_counts",
        color="predicted_doublet"
    )
    datasets[k] = _adata[~_adata.obs["predicted_doublet"], :].copy()

---
## 3. Normalization and Concatenation

In [ ]:
for k, _adata in datasets.items():
    _adata.layers["counts"] = _adata.X.copy()
    sc.pp.normalize_total(_adata, target_sum=1e4)
    sc.pp.log1p(_adata)
    datasets[k] = _adata


adata = ad.concat(datasets, label="batch_id")
adata.obs_names_make_unique()

---
## 4. Dimensionality Reduction and Manifold Embedding
### 4-1. Feature Selection

In [ ]:
ad_manifold = adata.copy()

sc.pp.highly_variable_genes(
    ad_manifold, min_mean=0.3, max_mean=4.5, min_disp=0.5
)
sc.pl.highly_variable_genes(ad_manifold, show=False)

ad_manifold = ad_manifold[:, ad_manifold.var.highly_variable]
sc.pp.scale(ad_manifold, max_value=10)

### 4-2. Dimensionality Reduction


In [ ]:
sc.tl.pca(ad_manifold, svd_solver="arpack")

### 4-3. Manifold Embedding

In [ ]:
sc.pp.neighbors(ad_manifold, n_pcs=50)
sc.tl.umap(ad_manifold)

In [ ]:

fig, ax = plt.subplots(figsize=(3, 3))

sc.pl.umap(ad_manifold, ax=ax, size=10, show=False)
ax.axis("off");

In [ ]:
transfer_embedding_info(adata_source=ad_manifold, adata_target=adata, overwrite=True)
del ad_manifold

---
## 5. Clustering

In [ ]:
sc.tl.leiden(adata, resolution=1)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))

sc.pl.umap(adata, ax=ax, size=10, show=False, color="leiden")
ax.axis("off")
ax.set(title=ENV_SUFFIX)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), title="Leiden", frameon=False)

fig.savefig(
    OUTPUT_DIR / f"gbm_umap_{ENV_SUFFIX}.pdf", **kwarg_savefig
)

---
## 6. Execution Environment Details

In [ ]:
%%bash
bash .rep_exe_env.sh